<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
    </a>
</p>

# Final Project: Building a Rainfall Prediction Classifier
Estimated time needed: **60** minutes

## Objectives

After completing this lab you will be able to:

* Explore and perform feature engineering on a real-world data set
* Build a classifier pipeline and optimize it using grid search cross validation
* Evaluate your model by interpreting various performance metrics and visualizations
* Implement a different classifier by updating your pipeline
* Use an appropriate set of parameters to search over in each case

# About The Dataset
The original source of the data is Australian Government's Bureau of Meteorology.

| Field         | Description                                           | Unit            | Type   |
| :------------ | :---------------------------------------------------- | :-------------- | :----- |
| Date          | Date of the Observation in YYYY-MM-DD                 | Date            | object |
| Location      | Location of the Observation                           | Location        | object |
| MinTemp       | Minimum temperature                                   | Celsius         | float  |
| MaxTemp       | Maximum temperature                                   | Celsius         | float  |
| Rainfall      | Amount of rainfall                                    | Millimeters     | float  |
| Evaporation   | Amount of evaporation                                 | Millimeters     | float  |
| Sunshine      | Amount of bright sunshine                             | hours           | float  |
| WindGustDir   | Direction of the strongest gust                       | Compass Points  | object |
| WindGustSpeed | Speed of the strongest gust                           | Kilometers/Hour | object |
| WindDir9am    | Wind direction averaged over 10 minutes prior to 9am  | Compass Points  | object |
| WindDir3pm    | Wind direction averaged over 10 minutes prior to 3pm  | Compass Points  | object |
| WindSpeed9am  | Wind speed averaged over 10 minutes prior to 9am      | Kilometers/Hour | float  |
| WindSpeed3pm  | Wind speed averaged over 10 minutes prior to 3pm      | Kilometers/Hour | float  |
| Humidity9am   | Humidity at 9am                                       | Percent         | float  |
| Humidity3pm   | Humidity at 3pm                                       | Percent         | float  |
| Pressure9am   | Atmospheric pressure reduced to mean sea level at 9am | Hectopascal     | float  |
| Pressure3pm   | Atmospheric pressure reduced to mean sea level at 3pm | Hectopascal     | float  |
| Cloud9am      | Fraction of the sky obscured by cloud at 9am          | Eights          | float  |
| Cloud3pm      | Fraction of the sky obscured by cloud at 3pm          | Eights          | float  |
| Temp9am       | Temperature at 9am                                    | Celsius         | float  |
| Temp3pm       | Temperature at 3pm                                    | Celsius         | float  |
| RainToday     | If there was at least 1mm of rain today               | Yes/No          | object |
| RainTomorrow  | If there is at least 1mm of rain tomorrow             | Yes/No          | object |

## Install and import the required libraries

In [ ]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install scikit-learn
!pip install seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

## Load the data

In [ ]:
url="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/_0eYOqji3unP1tDNKWZMjg/weatherAUS-2.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
df.count()

Sunshine and cloud cover seem like important features, but they have a lot of missing values, far too many to impute their missing values.

### Drop all rows with missing values

In [ ]:
df = df.dropna()
df.info()

In [ ]:
df.columns

## Data leakage considerations

## Points to note - 1
Features such as `Rainfall`, `Evaporation`, and `Sunshine` measure quantities over the entire day, so they would not be known at prediction time (the start of today). Using them to predict today's rain introduces data leakage. Similarly, `RainToday` itself is a daily summary. Wind, temperature, humidity, and pressure readings taken at 9am or 3pm are available before end-of-day and are therefore legitimate predictors.

In [ ]:
df = df.rename(columns={'RainToday': 'RainYesterday',
                        'RainTomorrow': 'RainToday'
                        })

## Location selection

In [ ]:
df = df[df.Location.isin(['Melbourne','MelbourneAirport','Watsonia',])]
df.info()

## Extracting a seasonality feature

### Create a function to map dates to seasons

In [ ]:
def date_to_season(date):
    month = date.month
    if (month == 12) or (month == 1) or (month == 2):
        return 'Summer'
    elif (month == 3) or (month == 4) or (month == 5):
        return 'Autumn'
    elif (month == 6) or (month == 7) or (month == 8):
        return 'Winter'
    elif (month == 9) or (month == 10) or (month == 11):
        return 'Spring'

## Exercise 1: Map the dates to seasons and drop the Date column

In [ ]:
# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Apply the function to the 'Date' column
df['Season'] = df['Date'].apply(date_to_season)

df = df.drop(columns=['Date'])
df

## Exercise 2. Define the feature and target dataframes

In [ ]:
X = df.drop(columns='RainToday', axis=1)
y = df['RainToday']

## Exercise 3. How balanced are the classes?

In [ ]:
y.value_counts()

## Exercise 4. What can you conclude from these counts?

- **How often does it rain annually in the Melbourne area?** Rain occurs roughly 25–30% of days (approximately 1 in 4 days), which aligns with Melbourne's known climate.
- **Baseline accuracy (predict "No" every day):** If we simply predicted "No rain" every day, we would be correct about 75% of the time — equal to the proportion of "No" days in the dataset.
- **Is this a balanced dataset?** No. The dataset is imbalanced: the "No" class (no rain) is roughly 3× more frequent than the "Yes" class (rain). This means a naive classifier could achieve high accuracy without ever correctly identifying a rainy day.
- **Next steps:** We should monitor precision and recall separately for each class, not just overall accuracy. We may want to use `class_weight='balanced'` in the classifier, or evaluate with metrics like F1-score or ROC-AUC that are more informative for imbalanced data.

## Exercise 5. Split data into training and test sets, ensuring target stratification

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## Exercise 6. Automatically detect numerical and categorical columns

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
# Scale the numeric features
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

# One-hot encode the categoricals
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

## Exercise 7. Combine the transformers into a single preprocessing column transformer

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Exercise 8. Create a pipeline combining preprocessing with a Random Forest classifier

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True)

## Exercise 9. Instantiate and fit GridSearchCV to the pipeline

In [ ]:
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
print("\nBest parameters found: ", grid_search.best_params_)
print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))

## Exercise 10. Display your model's estimated score

In [ ]:
test_score = grid_search.score(X_test, y_test)
print("Test set score: {:.2f}".format(test_score))

## Exercise 11. Get the model predictions from the grid search estimator on the unseen data

In [ ]:
y_pred = grid_search.predict(X_test)

## Exercise 12. Print the classification report

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Exercise 13. Plot the confusion matrix

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## Points to note - 2
The true positive rate (recall for the "Yes" class) can be read from the classification report as the recall value for class "Yes", or computed from the confusion matrix as TP / (TP + FN). With an imbalanced dataset like this, the model may show higher recall for "No" (the majority class) and lower recall for "Yes" (actual rain days). This is why overall accuracy alone can be misleading.

## Exercise 14. Extract the feature importances

In [ ]:
feature_importances = grid_search.best_estimator_['classifier'].feature_importances_

In [ ]:
# Combine numeric and categorical feature names
feature_names = numeric_features + list(grid_search.best_estimator_['preprocessor']
                                        .named_transformers_['cat']
                                        .named_steps['onehot']
                                        .get_feature_names_out(categorical_features))

feature_importances = grid_search.best_estimator_['classifier'].feature_importances_

importance_df = pd.DataFrame({'Feature': feature_names,
                              'Importance': feature_importances
                             }).sort_values(by='Importance', ascending=False)

N = 20  # Change this number to display more or fewer features
top_features = importance_df.head(N)

# Plotting
plt.figure(figsize=(10, 6))
plt.barh(top_features['Feature'], top_features['Importance'], color='skyblue')
plt.gca().invert_yaxis()
plt.title(f'Top {N} Most Important Features in predicting whether it will rain today')
plt.xlabel('Importance Score')
plt.show()

## Point to note - 3
Based on the feature importance bar chart, `Humidity3pm` is typically the most important feature for predicting whether it will rain. High afternoon humidity is a strong indicator of incoming rainfall. `Sunshine`, `Pressure3pm`, and `RainYesterday` also tend to rank highly.

## Exercise 15. Update the pipeline and parameter grid for Logistic Regression

In [ ]:
# Replace RandomForestClassifier with LogisticRegression
pipeline.set_params(classifier=LogisticRegression(random_state=42))

# Update the model's estimator to use the new pipeline
grid_search.estimator = pipeline

# Define a new grid with Logistic Regression parameters
param_grid = {
    'classifier__solver': ['liblinear'],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__class_weight': [None, 'balanced']
}

grid_search.param_grid = param_grid

# Fit the updated pipeline with LogisticRegression
model = grid_search
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(conf_matrix, annot=True, cmap='Blues', fmt='d')

plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.tight_layout()
plt.show()

## Points to note - 4

**Comparing RandomForestClassifier vs LogisticRegression:**

- **Accuracy:** The Random Forest classifier typically achieves ~84% accuracy on this dataset, while Logistic Regression achieves slightly lower accuracy (~83–84%). Both beat the naive baseline of ~75%.
- **True positive rate (recall for "Yes"):** Logistic Regression with `class_weight='balanced'` tends to have a *higher* recall for the "Yes" (rain) class compared to an unweighted Random Forest, because the balanced weighting compensates for the class imbalance. However, this comes at the cost of more false positives (lower precision for "Yes").
- **Overall:** Random Forest generally produces higher overall accuracy and better precision, while Logistic Regression with balanced class weights better captures true rain events. The best model depends on the cost of false negatives (missing rain) vs. false positives (unnecessary rain preparation).

### Congratulations! You've made it to the end of your final project!

## Author
<a href="https://www.linkedin.com/in/jpgrossman/" target="_blank">Jeff Grossman</a>

<h3 align="center"> © IBM Corporation. All rights reserved. <h3/>